The objective here will be to use a bitmap font atlas to match characters in the text field, and from that get an accurate OCR.

1. Load Atlas
2. Load Images
3. For each image:
   1. Prep a strip of the image for the date

In [1]:
from numpy import ndarray
from typing import Any

import numpy as np
import dask.array as da
import dask.dataframe as dd
import pandas as pd
import dask
from dask_image import ndfilters, ndmeasure, ndmorph

# from dask.distributed import Client
# if 'client' not in locals():
#     client = Client()

from PIL import Image
from dask.array.image import imread
filename_pattern = r'/home/rainybyte/AllSkyImages/2010-09/*.JPG'
images = imread(filename_pattern)
images

dask.array<imread, shape=(11696, 480, 640, 3), dtype=uint8, chunksize=(1, 480, 640, 3), chunktype=numpy.ndarray>

In [2]:
red = images.transpose(0, 3, 1, 2)[0][0]
red

dask.array<getitem, shape=(480, 640), dtype=uint8, chunksize=(480, 640), chunktype=numpy.ndarray>

In [3]:
# These are specific to my atlas
atlas_chars = "0123456789s"
atlas_symbs = ":/."
atlas_image = imread('/home/rainybyte/AllSkyImages/font_atlas.bmp') # 66 x 16 pixels
char_width = 6
char_height = 8
symb_width = 3
symb_height = 8

# Now split into two atlas char arrays, one for the characters at 6x8 pixels and one for symbols at 3x8
# Note: the symbol atlas is not used in the current implementation, as we know where all the symbols should be and can fail the parse in the cases when we are wrong.
char_atlas = atlas_image[0][:8, :]

atlas_charcount = char_atlas.shape[1] // char_width

# Reshape into (8, chars, char_width)
char_atlas_chars = char_atlas.reshape(8, atlas_charcount, char_width).transpose(1, 0, 2).astype(np.float32).compute()
char_glyphs = dict(zip(atlas_chars, char_atlas_chars))

templates = np.stack([char_glyphs[ch] for ch in atlas_chars]).astype(np.float32)
templates = templates - templates.mean(axis=(1, 2), keepdims=True)
template_norms = np.sqrt(np.sum(templates * templates, axis=(1, 2))) + 1e-6
templates = templates.reshape(templates.shape[0], -1)

In [4]:
def score_patch_against_template(patch, template, eps=1e-6):
    """
    Returns a normalized correlation score between a patch and a glyph template.
    Higher is better.
    """
    patch = np.asarray(patch, dtype=np.float32)
    template = np.asarray(template, dtype=np.float32)

    patch = patch - patch.mean()
    template = template - template.mean()

    denom = np.sqrt(np.sum(patch ** 2) * np.sum(template ** 2)) + eps
    return float(np.sum(patch * template) / denom)


def classify_at_cursor(image, x, y, atlas, atlas_chars, width, height, eps=1e-6):
    """
    Classify the glyph centered/anchored at a known cursor position.

    Parameters
    ----------
    image : ndarray
        2D grayscale image.
    x, y : int
        Top-left position of the glyph box in the image.
    atlas : dict[str, ndarray]
        Mapping from character label to glyph bitmap.
    atlas_chars : iterable[str]
        Characters to test.
    width, height : int
        Glyph size.
    eps : float
        Small constant for numerical stability.

    Returns
    -------
    best_char : str
    best_score : float
    scores : dict[str, float]
    """
    patch = image[y:y + height, x:x + width]
    if patch.shape != (height, width):
        raise ValueError(f"Patch shape {patch.shape} does not match {(height, width)}")

    scores = {}
    for ch in atlas_chars:
        scores[ch] = score_patch_against_template(patch, atlas[ch], eps=eps)

    best_char = max(scores, key=scores.get)
    best_score = scores[best_char]
    return best_char, best_score, scores

Now that we have all the glyphs set up, we can classify the characters in the aliased-text images

In [5]:
best_char, best_score, scores = classify_at_cursor(
    image=red.astype(np.float32),
    x=5+char_width*1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('0',
 0.9999374151229858,
 {'0': 0.9999374151229858,
  '1': -0.07989895343780518,
  '2': 0.4065017104148865,
  '3': 0.6845079660415649,
  '4': -0.08787599205970764,
  '5': 0.5073401927947998,
  '6': 0.7827118039131165,
  '7': -0.05048016086220741,
  '8': 0.783406138420105,
  '9': 0.7795872688293457,
  's': 0.20379485189914703})

Below we can see that this causes issues with the antialiased images.

In [6]:
images_2019 = imread(r'/home/rainybyte/AllSkyImages/2019-09/*.JPG')
red_2019 = images_2019.transpose(0, 3, 1, 2)[0][0]

best_char, best_score, scores = classify_at_cursor(
    image=red_2019.astype(np.float32),
    x=5+(char_width)*1,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=6,
    height=8,
)
best_char, best_score, scores

('5',
 0.36639389395713806,
 {'0': 0.33899861574172974,
  '1': -0.06114104762673378,
  '2': 0.2295859158039093,
  '3': 0.23315724730491638,
  '4': -0.165544331073761,
  '5': 0.36639389395713806,
  '6': 0.29352709650993347,
  '7': 0.020206689834594727,
  '8': 0.2833721339702606,
  '9': 0.25677579641342163,
  's': 0.02235008217394352})

In [7]:
def extract_patches_2d(image, positions, width, height):
    patches = []
    for x, y in positions:
        patch = np.asarray(image[y:y + height, x:x + width], dtype=np.float32)
        if patch.shape != (height, width):
            raise ValueError(f"Bad patch shape {patch.shape}, expected {(height, width)}")
        patch = patch - patch.mean()
        patches.append(patch.reshape(-1))
    return np.stack(patches, axis=0)

def classify_patches_2d(patches_2d, templates_2d, template_norms):
    patch_norms = np.linalg.norm(patches_2d, axis=1) + 1e-6
    raw = patches_2d @ templates_2d.T
    scores = raw / (patch_norms[:, None] * template_norms[None, :])

    best_idx = np.argmax(scores, axis=1)
    best_scores = scores[np.arange(scores.shape[0]), best_idx]
    return best_idx, best_scores, scores


def classify_patch_fast(patch, templates, template_norms, chars):
    patch = patch.astype(np.float32, copy=False)
    patch = patch - patch.mean()
    patch_norm = np.sqrt(np.sum(patch * patch)) + 1e-6

    # flatten once
    p = patch.ravel()
    t = templates

    scores = (t @ p) / (template_norms * patch_norm)
    idx = np.argmax(scores)
    return chars[idx], float(scores[idx]), scores

# Represents the positions "XXXX/XX/XX" in the top-left, first row
DATE_POS = [
    (5 + 0 * char_width, 6),
    (5 + 1 * char_width, 6),
    (5 + 2 * char_width, 6),
    (5 + 3 * char_width, 6),
    (5 + 4 * char_width + symb_width, 6),
    (5 + 5 * char_width + symb_width, 6),
    (5 + 6 * char_width + 2 * symb_width, 6),
    (5 + 7 * char_width + 2 * symb_width, 6),
]

# Represents the positions "XX:XX:XX" in the top-left, second row
TIME_POS = [
    (5 + 0 * char_width, 19),
    (5 + 1 * char_width, 19),
    (20 + 0 * char_width, 19),
    (20 + 1 * char_width, 19),
    (35 + 0 * char_width, 19),
    (35 + 1 * char_width, 19),
]

# TODO: EXPOSURE TIME CAN"T BE DETERMINISTIC.

# Represents the positions "XXXXXXXXX" in the bottom-left.
FILENAME_POS = [
    (5 + 0 * char_width, 466),
    (5 + 1 * char_width, 466),
    (5 + 2 * char_width, 466),
    (5 + 3 * char_width, 466),
    (5 + 4 * char_width, 466),
    (5 + 5 * char_width, 466),
    (5 + 6 * char_width, 466),
    (5 + 7 * char_width, 466),
    (5 + 8 * char_width, 466),
]

# def classify_date_string_fast(
#         image,
#         tolerance=0.85,
# ):
#     image = image.reshape((480,640))
#
#     matched_chars = []
#     for (x, y) in DATE_POS:
#         patch = image[y:y + char_height, x:x + char_width]
#         best_char, best_score, scores = classify_patch_fast(patch, templates, template_norms, atlas_chars)
#         if best_score < tolerance:
#             best_char = '?'
#             best_score = None
#         matched_chars.append(best_char)
#     matched_chars = matched_chars[:4] + ['/'] + matched_chars[4:6] + ['/'] + matched_chars[6:]
#     return ''.join(matched_chars)

def classify_patches_deterministic(image, positions, tolerance: float):
    patches_2d = extract_patches_2d(image, positions, char_width, char_height)
    best_idx, best_scores, _ = classify_patches_2d(patches_2d, templates, template_norms)

    chars = []
    for idx, score in zip(best_idx, best_scores):
        ch = atlas_chars[idx]
        if score < tolerance:
            ch = '?'
        chars.append(ch)
    return chars


def classify_date_string_fast(image, tolerance=0.85):
    image = np.asarray(image, dtype=np.float32).reshape(480, 640)
    chars = classify_patches_deterministic(image, DATE_POS, tolerance)

    chars = chars[:4] + ['/'] + chars[4:6] + ['/'] + chars[6:]
    return ''.join(chars)

def classify_time_string_fast(image, tolerance=0.85):
    image = np.asarray(image, dtype=np.float32).reshape(480, 640)
    chars = classify_patches_deterministic(image, TIME_POS, tolerance)
    chars = chars[:2] + [':'] + chars[2:4] + [':'] + chars[4:]
    return ''.join(chars)

def classify_filename_string_fast(image, tolerance=0.85):
    image = np.asarray(image, dtype=np.float32).reshape(480, 640)
    chars = classify_patches_deterministic(image, FILENAME_POS, tolerance)
    return ''.join(chars)

def classify_date_string_fast_block(block):
    results = []
    for img in block:
        results.append(classify_date_string_fast(img))
    return np.array(results, dtype=str)

def classify_fields_block(block):
    results = []
    for img in block:
        results.append({
            'date': classify_date_string_fast(img),
            'time': classify_time_string_fast(img),
            'exposure': None,
            'filename': classify_filename_string_fast(img),
            # TODO: fields related to the image content
        })
    return pd.DataFrame.from_records(results)

In [8]:
classify_date_string_fast(red)

'2010/09/01'

In [9]:
batch = images.transpose(0, 3, 1, 2)[1000:1025]
batch_reds = batch[:,0]

In [10]:
[classify_date_string_fast(image) for image in batch_reds]

['2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03',
 '2010/09/03']

In [11]:
batch_reds.map_blocks(classify_date_string_fast_block, dtype=str, drop_axis=(1,2)).compute()

array(['2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03', '2010/09/03', '2010/09/03', '2010/09/03',
       '2010/09/03'], dtype='<U10')

In [12]:
all_batch = images.transpose(0, 3, 1, 2)
all_batch_reds = all_batch[:,0].rechunk((32,480,640,3))

In [13]:
all_parsed: np.ndarray = all_batch_reds.map_blocks(classify_date_string_fast_block, dtype=str, drop_axis=(1,2)).compute().astype(str)
all_parsed

array(['2010/09/01', '2010/09/01', '2010/09/01', ..., '2010/10/01',
       '2010/10/01', '2010/10/01'], shape=(11696,), dtype='<U10')

In [14]:
mask = np.char.find(all_parsed, '?') != -1

In [15]:
np.any(np.char.find(all_parsed, 's') != -1)

np.False_

In [16]:
np.count_nonzero(mask)

np.int64(0)

In [17]:
pd.DataFrame(all_parsed[mask])

,0


In [18]:
pd.DataFrame(all_parsed[~mask])

,0
0,2010/09/01
1,2010/09/01
2,2010/09/01
3,2010/09/01
4,2010/09/01
...,...
11691,2010/10/01
11692,2010/10/01
11693,2010/10/01
11694,2010/10/01


In [19]:
# pd.DataFrame(
#     list(all_batch_reds.map_blocks(classify_fields_block, dtype=object, drop_axis=(1,2)).compute()),
# ).to_parquet('../../data/test.parquet')

In [20]:
df = all_batch_reds.map_blocks(classify_fields_block, dtype=object, drop_axis=(1,2)).compute()
df

,date,time,exposure,filename
0,2010/09/01,00:58:36,None,000029734
1,2010/09/01,01:00:18,None,000029735
2,2010/09/01,01:01:47,None,000029736
3,2010/09/01,01:03:26,None,000029737
4,2010/09/01,01:04:46,None,000029738
...,...,...,...,...
11,2010/10/01,00:50:39,None,000041601
12,2010/10/01,00:52:43,None,000041602
13,2010/10/01,00:54:08,None,000041603
14,2010/10/01,00:55:53,None,000041604


In [21]:
import os
from glob import glob

image_folders = sorted(glob("/home/rainybyte/AllSkyImages/20*/"))
len(image_folders), image_folders

(100,
 ['/home/rainybyte/AllSkyImages/2010-08/',
  '/home/rainybyte/AllSkyImages/2010-09/',
  '/home/rainybyte/AllSkyImages/2010-10/',
  '/home/rainybyte/AllSkyImages/2010-11/',
  '/home/rainybyte/AllSkyImages/2010-12/',
  '/home/rainybyte/AllSkyImages/2011-01/',
  '/home/rainybyte/AllSkyImages/2011-02/',
  '/home/rainybyte/AllSkyImages/2011-03/',
  '/home/rainybyte/AllSkyImages/2011-04/',
  '/home/rainybyte/AllSkyImages/2011-05/',
  '/home/rainybyte/AllSkyImages/2011-06/',
  '/home/rainybyte/AllSkyImages/2011-07/',
  '/home/rainybyte/AllSkyImages/2011-08/',
  '/home/rainybyte/AllSkyImages/2011-09/',
  '/home/rainybyte/AllSkyImages/2011-10/',
  '/home/rainybyte/AllSkyImages/2011-11/',
  '/home/rainybyte/AllSkyImages/2011-12/',
  '/home/rainybyte/AllSkyImages/2012-01/',
  '/home/rainybyte/AllSkyImages/2012-02/',
  '/home/rainybyte/AllSkyImages/2012-03/',
  '/home/rainybyte/AllSkyImages/2012-04/',
  '/home/rainybyte/AllSkyImages/2012-05/',
  '/home/rainybyte/AllSkyImages/2012-06/',
  '/h

In [22]:
def process_allsky_image_folder(directory: str,):
    print(directory)
    filename_pattern = f'{directory}*.JPG'
    images = imread(filename_pattern)
    # TODO: see if I can rework the downstream code to not need this transpose
    all_batch_reds = images.transpose(0, 3, 1, 2)[:,0].rechunk((32,480,640))
    df = all_batch_reds.map_blocks(classify_fields_block, dtype=object, drop_axis=(1,2)).compute()
    df.to_parquet(f'../../data/{os.path.basename(directory[:-1])}.parquet')

In [23]:
tasks = [dask.delayed(process_allsky_image_folder)(folder) for folder in sorted(image_folders)]
dask.compute(tasks, scheduler='processes', progress=True)

/home/rainybyte/AllSkyImages/2019-04/
/home/rainybyte/AllSkyImages/2011-03/
/home/rainybyte/AllSkyImages/2019-06/
/home/rainybyte/AllSkyImages/2014-01/
/home/rainybyte/AllSkyImages/2014-12/
/home/rainybyte/AllSkyImages/2015-05/
/home/rainybyte/AllSkyImages/2018-02/
/home/rainybyte/AllSkyImages/2019-12/
/home/rainybyte/AllSkyImages/2018-11/
/home/rainybyte/AllSkyImages/2015-08/
/home/rainybyte/AllSkyImages/2012-05/
/home/rainybyte/AllSkyImages/2010-09/
/home/rainybyte/AllSkyImages/2015-11/
/home/rainybyte/AllSkyImages/2014-03/
/home/rainybyte/AllSkyImages/2011-09/
/home/rainybyte/AllSkyImages/2018-10/
/home/rainybyte/AllSkyImages/2011-05/
/home/rainybyte/AllSkyImages/2020-01/
/home/rainybyte/AllSkyImages/2015-02/
/home/rainybyte/AllSkyImages/2015-12/
/home/rainybyte/AllSkyImages/2012-10/
/home/rainybyte/AllSkyImages/2014-10/
/home/rainybyte/AllSkyImages/2014-05/
/home/rainybyte/AllSkyImages/2019-10/
/home/rainybyte/AllSkyImages/2010-11/
/home/rainybyte/AllSkyImages/2011-08/
/home/rainyb

OSError: image file is truncated (85 bytes not processed)